# Talos — linear static FEA in a notebook

Talos is standalone: it only needs a STEP file (from any CAD program), Gmsh and CalculiX.
Here we create a simple STEP with Gmsh itself so the notebook does not depend on Dedalus.
Each step (inspect, mesh, solve) is started explicitly.

In [ ]:
from pathlib import Path
import gmsh
from vegeta import talos

RUNS = Path("_runs/talos"); RUNS.mkdir(parents=True, exist_ok=True)
step = RUNS / "beam.step"
gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
gmsh.model.occ.addBox(0, -10, -5, 200, 20, 10); gmsh.model.occ.synchronize()
gmsh.write(str(step)); gmsh.finalize()

## 1. Inspect the geometry to choose regions

In [ ]:
info = talos.inspect_step(step, units="mm-N-MPa")
info

## 2. Define the analysis explicitly
Material values are the engineer's responsibility; none are built in.

In [ ]:
steel = talos.Material("S235", youngs_modulus=210000, poissons_ratio=0.3,
                       density=7.85e-9, yield_strength=235, source="example values")
model = talos.StructuralModel(
    geometry=step, units="mm-N-MPa", material=steel,
    regions=[talos.SurfacesOnPlane("fixed", "x", 0.0), talos.SurfacesOnPlane("tip", "x", 200.0)],
    supports=[talos.FixedSupport("fixed")],
    loads=[talos.Force("tip", fz=-100.0)],
    mesh_settings=talos.MeshSettings(element_size=5.0, order=2),
)

## 3. Mesh (Gmsh)

In [ ]:
mesh_res = model.mesh(RUNS / "cantilever", progress=True)
mesh_res

## 4. Solve (CalculiX)

In [ ]:
res = model.solve(RUNS / "cantilever", progress=True)
res

Compare with Euler–Bernoulli beam theory:

In [ ]:
E, L, b, h, F = 210000, 200, 20, 10, 100
I = b * h**3 / 12
print("beam theory tip deflection:", F * L**3 / (3 * E * I))
print("Talos tip deflection:     ", -res.metrics["displacement_min"][2] if res.ok else res.messages)

In [ ]:
if res.ok:
    fig1 = talos.plot_deformed(res)
    fig2 = talos.plot_along_axis(res, axis="x", quantity="displacement", component=2)

## 5. Change something and re-run — into a new directory, history is kept

In [ ]:
import dataclasses
model2 = dataclasses.replace(model, loads=[talos.Force("tip", fz=-100.0), talos.Acceleration(az=-9810.0)])
model2.mesh(RUNS / "cantilever_gravity")
res2 = model2.solve(RUNS / "cantilever_gravity")
res2.metrics.get("max_displacement"), res.metrics.get("max_displacement")

Solving never meshes implicitly — an out-of-date mesh is reported, not silently regenerated:

In [ ]:
model3 = dataclasses.replace(model, mesh_settings=talos.MeshSettings(element_size=3.0))
model3.solve(RUNS / "cantilever").messages